In [1]:
import pandas as pd
import re
import json
import math
from difflib import SequenceMatcher
from ollama import chat, ChatResponse
from tqdm import tqdm

In [2]:
# =========================
# Settings
# =========================
FILE_PATH = "./test/RACE_High_Combined.xlsx"

# 依序跑完一個模型才換下一個（效率較好）
MODELS = [
    ("gemma3:12b", False),        # (model_name, think)
    ("qwen3:8b", False),
    ("deepseek-r1:8b", False),
]

# deepseek thinking mode（只用在 rule6 fallback）
DEEPSEEK_THINK_MODEL = "deepseek-r1:8b"
DEEPSEEK_THINK = True

CONTEXT_LEN = 1536
TEMPERATURE = 0.0

# 單模型輸出檔
OUT_MODEL_FILE = {
    "gemma3:12b": "result_gemma3_12b.xlsx",
    "qwen3:8b": "result_qwen3_8b.xlsx",
    "deepseek-r1:8b": "result_deepseek_r1_8b.xlsx",
}

# Ensemble 輸出檔
OUT_ENSEMBLE_FILE = "result_ensemble.xlsx"

# Accuracy summary 輸出檔
OUT_SUMMARY_XLSX = "accuracy_summary.xlsx"
OUT_SUMMARY_CSV = "accuracy_summary.csv"

In [3]:
# =========================
# Ollama call
# =========================
def query_ollama(prompt, model, context_len=CONTEXT_LEN, think=False) -> str:
    try:
        resp: ChatResponse = chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={"num_ctx": context_len, "temperature": TEMPERATURE},
            think=think
        )
        return resp.message.content or ""
    except Exception as e:
        print(f"[Ollama Error] model={model} err={e}")
        return ""

In [4]:
# =========================
# Utility: safe cast / NaN handling
# =========================
def is_nan(x) -> bool:
    try:
        return x is not None and isinstance(x, float) and math.isnan(x)
    except Exception:
        return False

def to_int_1_4(x):
    """
    將 x 轉成 1~4 的 int；不合法（含 NaN/None/其他）回傳 None
    """
    if x is None:
        return None
    # pandas/np 的 NaN
    if pd.isna(x):
        return None
    # 有些會是 1.0 這種 float
    try:
        xi = int(x)
    except Exception:
        # 可能是字串
        try:
            s = str(x).strip()
            if s in ["1", "2", "3", "4"]:
                return int(s)
            return None
        except Exception:
            return None
    return xi if xi in (1, 2, 3, 4) else None

def to_conf_0_1(x):
    """
    將 confidence 轉成 0~1 float；不合法（含 NaN/None）回傳 0.0
    """
    if x is None or pd.isna(x):
        return 0.0
    try:
        xf = float(x)
    except Exception:
        return 0.0
    if math.isnan(xf):
        return 0.0
    return max(0.0, min(1.0, xf))

In [5]:
# =========================
# Parsing helpers
# =========================
def _strip_think(text: str) -> str:
    if not text:
        return ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def _normalize(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[\r\n\t]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^a-z0-9\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [6]:
def _best_option_match(text_cleaned: str, options: dict) -> int | None:
    t = _normalize(text_cleaned)
    if not t:
        return None

    for k, opt in options.items():
        o = _normalize(opt)
        if o and (o in t or t in o):
            return int(k)

    best_k, best_score = None, 0.0
    for k, opt in options.items():
        o = _normalize(opt)
        if not o:
            continue
        score = SequenceMatcher(None, t, o).ratio()
        if score > best_score:
            best_k, best_score = k, score

    if best_k is not None and best_score >= 0.55:
        return int(best_k)
    return None

In [7]:
def parse_answer_and_confidence(raw_text: str, options: dict) -> tuple[int | None, float | None]:
    """
    解析 LLM 輸出：
    - 優先 JSON: {"answer":1,"confidence":0.82}
    - 其次 Answer: X / Confidence: 0.xx
    - 再不行找最後的 1-4
    - 最後用 options 比對文字
    """
    if not raw_text:
        return None, None

    text = _strip_think(raw_text)

    # A) JSON
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if m:
        try:
            obj = json.loads(m.group(0))
            ans = to_int_1_4(obj.get("answer", None))
            conf = obj.get("confidence", None)
            conf = None if conf is None else float(conf)
            if conf is not None:
                conf = max(0.0, min(1.0, conf))
            if ans is not None:
                return ans, conf
        except Exception:
            pass

    # B) Answer / Confidence
    m1 = re.search(r"Answer:\s*([1-4])", text, re.IGNORECASE)
    m2 = re.search(r"Confidence:\s*([0-9]*\.?[0-9]+)", text, re.IGNORECASE)
    ans = int(m1.group(1)) if m1 else None
    conf = float(m2.group(1)) if m2 else None
    if conf is not None:
        conf = max(0.0, min(1.0, conf))
    if ans is not None:
        return ans, conf

    # C) fallback: last digit 1-4
    nums = re.findall(r"\b([1-4])\b", text)
    if nums:
        return int(nums[-1]), conf

    # D) options match
    matched = _best_option_match(text, options)
    return matched, conf

In [8]:
# =========================
# Prompt builder
# =========================
def build_prompt(article: str, question: str, valid_options: dict) -> str:
    options_text = "\n".join([f"{k}. {v}" for k, v in valid_options.items()])
    valid_numbers = ", ".join([str(k) for k in valid_options.keys()])
    return f"""
Read the following article and answer the multiple-choice question.

Article:
{article}

Question:
{question}

Options:
{options_text}

Instruction:
1) Select the best option among ({valid_numbers}). If an option is missing/empty, do not select it.
2) Output ONLY a JSON object with:
   - "answer": an integer in [{valid_numbers}]
   - "confidence": a number in [0, 1] representing your probability the answer is correct.

Example:
{{"answer": 2, "confidence": 0.78}}
""".strip()

In [9]:
# =========================
# Ensemble helpers (safe)
# =========================
def majority_vote(preds: dict) -> int | None:
    counts = {}
    for a in preds.values():
        a = to_int_1_4(a)
        if a is None:
            continue
        counts[a] = counts.get(a, 0) + 1
    if not counts:
        return None
    best_a = max(counts, key=counts.get)
    top = [k for k, v in counts.items() if v == counts[best_a]]
    return None if len(top) > 1 else best_a

In [10]:
def weighted_vote(preds: dict, confs: dict) -> int | None:
    scores = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    total_mass = 0.0

    for m, a in preds.items():
        a = to_int_1_4(a)
        if a is None:
            continue
        w = to_conf_0_1(confs.get(m, None))
        scores[a] += w
        total_mass += w

    if total_mass <= 0.0:
        return None

    best_a = max(scores, key=lambda k: scores[k])
    best_score = scores[best_a]
    tied = [k for k, v in scores.items() if abs(v - best_score) < 1e-12]
    return None if len(tied) > 1 else best_a

In [11]:
# =========================
# Step 1: Run one model over whole dataset
# =========================
def run_single_model(df: pd.DataFrame, model_name: str, think: bool) -> tuple[str, int, int, float]:
    print(f"\n====== Running {model_name} (think={think}) ======")

    correct = 0
    results = []
    total = len(df)

    for _, row in tqdm(df.iterrows(), total=total, desc=f"Predicting[{model_name}]"):
        options_all = {
            1: row["Option1"],
            2: row["Option2"],
            3: row["Option3"],
            4: row["Option4"],
        }
        valid_options = {k: v for k, v in options_all.items() if v and str(v).lower() != "nan"}

        prompt = build_prompt(row["Article"], row["Question"], valid_options)
        raw = query_ollama(prompt, model=model_name, think=think)

        pred, conf = parse_answer_and_confidence(raw, valid_options)
        pred = to_int_1_4(pred)

        actual = to_int_1_4(row["Answer"])
        is_correct = (actual is not None and pred is not None and pred == actual)
        if is_correct:
            correct += 1

        results.append({
            "ID": row["ID"],
            "Actual": actual,
            "Predicted": pred,
            "Confidence": conf,
            "Correct": is_correct,
            "Model_Response": raw
        })

    acc = (correct / total * 100) if total else 0.0
    print(f"{model_name} Accuracy: {correct}/{total} = {acc:.2f}%")

    out_file = OUT_MODEL_FILE.get(model_name, f"result_{model_name.replace(':', '_')}.xlsx")
    pd.DataFrame(results).to_excel(out_file, index=False)
    print(f"[Saved] {out_file}")

    return out_file, correct, total, acc

In [12]:
# =========================
# Step 2: Ensemble from saved files (no LLM calls except deepseek thinking fallback)
# =========================
def run_ensemble(df_base: pd.DataFrame, file_gemma: str, file_qwen: str, file_deepseek: str):
    df_g = pd.read_excel(file_gemma)
    df_q = pd.read_excel(file_qwen)
    df_d = pd.read_excel(file_deepseek)

    total = len(df_base)
    correct_major = 0
    correct_weight = 0
    correct_rule6 = 0

    rows = []

    for i in tqdm(range(total), desc="Ensembling"):
        row = df_base.iloc[i]

        options_all = {
            1: row["Option1"],
            2: row["Option2"],
            3: row["Option3"],
            4: row["Option4"],
        }
        valid_options = {k: v for k, v in options_all.items() if v and str(v).lower() != "nan"}
        prompt = build_prompt(row["Article"], row["Question"], valid_options)

        actual = to_int_1_4(df_g.loc[i, "Actual"])

        preds = {
            "gemma3:12b": df_g.loc[i, "Predicted"],
            "qwen3:8b": df_q.loc[i, "Predicted"],
            "deepseek-r1:8b": df_d.loc[i, "Predicted"],
        }
        confs = {
            "gemma3:12b": df_g.loc[i, "Confidence"],
            "qwen3:8b": df_q.loc[i, "Confidence"],
            "deepseek-r1:8b": df_d.loc[i, "Confidence"],
        }

        maj = majority_vote(preds)
        maj_correct = (actual is not None and maj is not None and maj == actual)
        if maj_correct:
            correct_major += 1

        wv = weighted_vote(preds, confs)
        wv_correct = (actual is not None and wv is not None and wv == actual)
        if wv_correct:
            correct_weight += 1

        # rule6
        gemma_ans = to_int_1_4(preds["gemma3:12b"])
        qwen_ans = to_int_1_4(preds["qwen3:8b"])

        deepseek_think_raw = None
        deepseek_think_pred = None
        deepseek_think_conf = None

        if gemma_ans is not None and gemma_ans == qwen_ans:
            rule6_pred = gemma_ans
        else:
            deepseek_think_raw = query_ollama(prompt, model=DEEPSEEK_THINK_MODEL, think=DEEPSEEK_THINK)
            deepseek_think_pred, deepseek_think_conf = parse_answer_and_confidence(deepseek_think_raw, valid_options)
            deepseek_think_pred = to_int_1_4(deepseek_think_pred)
            rule6_pred = deepseek_think_pred if deepseek_think_pred is not None else qwen_ans

        rule6_correct = (actual is not None and rule6_pred is not None and rule6_pred == actual)
        if rule6_correct:
            correct_rule6 += 1

        rows.append({
            "ID": df_g.loc[i, "ID"],
            "Actual": actual,

            "Gemma_Pred": gemma_ans,
            "Gemma_Conf": to_conf_0_1(confs["gemma3:12b"]),
            "Qwen_Pred": qwen_ans,
            "Qwen_Conf": to_conf_0_1(confs["qwen3:8b"]),
            "DeepSeek_Pred": to_int_1_4(preds["deepseek-r1:8b"]),
            "DeepSeek_Conf": to_conf_0_1(confs["deepseek-r1:8b"]),

            "MajorityVote": maj,
            "MajorityVote_Correct": maj_correct,
            "WeightedVote": wv,
            "WeightedVote_Correct": wv_correct,

            "Rule6_Final": rule6_pred,
            "Rule6_Correct": rule6_correct,

            "DeepSeekThink_Pred": deepseek_think_pred,
            "DeepSeekThink_Conf": deepseek_think_conf,
            "DeepSeekThink_Response": deepseek_think_raw
        })

    maj_acc = correct_major / total * 100 if total else 0.0
    wv_acc = correct_weight / total * 100 if total else 0.0
    rule6_acc = correct_rule6 / total * 100 if total else 0.0

    print("\n" + "=" * 40)
    print(f"Majority Accuracy: {correct_major}/{total} = {maj_acc:.2f}%")
    print(f"Weighted Accuracy: {correct_weight}/{total} = {wv_acc:.2f}%")
    print(f"Rule6 Accuracy: {correct_rule6}/{total} = {rule6_acc:.2f}%")
    print("=" * 40)

    pd.DataFrame(rows).to_excel(OUT_ENSEMBLE_FILE, index=False)
    print(f"[Saved] {OUT_ENSEMBLE_FILE}")

    return (correct_major, maj_acc), (correct_weight, wv_acc), (correct_rule6, rule6_acc)

In [13]:
# =========================
# Entry
# =========================
print(f"Loading dataset: {FILE_PATH}")
df = pd.read_excel(FILE_PATH)

required_columns = ["ID", "Article", "Question", "Option1", "Option2", "Option3", "Option4", "Answer"]
if not all(c in df.columns for c in required_columns):
    raise ValueError(f"Excel 缺少必要欄位，需要: {required_columns}")

Loading dataset: ./test/RACE_High_Combined.xlsx


In [14]:
outputs = {}
summary_rows = []

# 1) 各模型跑完整 dataset
for model_name, think in MODELS:
    out_file, correct, total, acc = run_single_model(df, model_name, think)
    outputs[model_name] = out_file
    summary_rows.append({
        "Method": model_name,
        "Correct": correct,
        "Total": total,
        "Accuracy(%)": round(acc, 4)
    })


====== Running gemma3:12b (think=False) ======


Predicting[gemma3:12b]: 100%|██████████████████████████████████████████████████████| 3498/3498 [40:48<00:00,  1.43it/s]


gemma3:12b Accuracy: 2984/3498 = 85.31%
[Saved] result_gemma3_12b.xlsx

====== Running qwen3:8b (think=False) ======


Predicting[qwen3:8b]: 100%|████████████████████████████████████████████████████████| 3498/3498 [15:34<00:00,  3.74it/s]


qwen3:8b Accuracy: 3040/3498 = 86.91%
[Saved] result_qwen3_8b.xlsx

====== Running deepseek-r1:8b (think=False) ======


Predicting[deepseek-r1:8b]: 100%|██████████████████████████████████████████████████| 3498/3498 [37:00<00:00,  1.58it/s]


deepseek-r1:8b Accuracy: 2962/3498 = 84.68%
[Saved] result_deepseek_r1_8b.xlsx


In [15]:
# 2) ensemble
(maj_correct, maj_acc), (wv_correct, wv_acc), (r6_correct, r6_acc) = run_ensemble(
    df_base=df,
    file_gemma=outputs["gemma3:12b"],
    file_qwen=outputs["qwen3:8b"],
    file_deepseek=outputs["deepseek-r1:8b"],
)

summary_rows.extend([
    {"Method": "Ensemble-Majority (3 models)", "Correct": maj_correct, "Total": len(df), "Accuracy(%)": round(maj_acc, 4)},
    {"Method": "Ensemble-Weighted (3 models)", "Correct": wv_correct, "Total": len(df), "Accuracy(%)": round(wv_acc, 4)},
    {"Method": "Rule6 (Gemma==Qwen else DeepSeek-Think else Qwen)", "Correct": r6_correct, "Total": len(df), "Accuracy(%)": round(r6_acc, 4)},
])

summary_df = pd.DataFrame(summary_rows).sort_values(by="Accuracy(%)", ascending=False)
summary_df.to_excel(OUT_SUMMARY_XLSX, index=False)
summary_df.to_csv(OUT_SUMMARY_CSV, index=False, encoding="utf-8-sig")

print(f"\n[Saved] {OUT_SUMMARY_XLSX}")
print(f"[Saved] {OUT_SUMMARY_CSV}")
print("\n===== Accuracy Summary =====")
print(summary_df.to_string(index=False))

Ensembling: 100%|████████████████████████████████████████████████████████████████| 3498/3498 [2:24:16<00:00,  2.47s/it]



Majority Accuracy: 3041/3498 = 86.94%
Weighted Accuracy: 3040/3498 = 86.91%
Rule6 Accuracy: 3091/3498 = 88.36%
[Saved] result_ensemble.xlsx

[Saved] accuracy_summary.xlsx
[Saved] accuracy_summary.csv

===== Accuracy Summary =====
                                           Method  Correct  Total  Accuracy(%)
Rule6 (Gemma==Qwen else DeepSeek-Think else Qwen)     3091   3498      88.3648
                     Ensemble-Majority (3 models)     3041   3498      86.9354
                                         qwen3:8b     3040   3498      86.9068
                     Ensemble-Weighted (3 models)     3040   3498      86.9068
                                       gemma3:12b     2984   3498      85.3059
                                   deepseek-r1:8b     2962   3498      84.6770
